# Support Vector Machines (SVM) — Wine Dataset

## Objective

The goal of this experiment is to train and evaluate a **Support Vector Machine (SVM)** classifier on the **Wine dataset**, and to analyze how different SVM hyperparameters affect the model’s performance.

Specifically, we investigate the impact of:

* Regularization parameter **C**
* Kernel type (**linear**, **rbf**, **poly**)
* Kernel coefficient **gamma**


In [1]:
import pandas as pd
import numpy as np
import os 

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

## Dataset Description

The Wine dataset contains physicochemical properties of wines derived from three different cultivars.

**Target variable**

* `Wine`: Class label (1, 2, 3)

**Input features**

* Alcohol
* Malic acid
* Ash
* Alcalinity of ash
* Magnesium
* Total phenols
* Flavanoids
* Nonflavanoid phenols
* Proanthocyanins
* Color intensity
* Hue
* OD280/OD315 of diluted wines
* Proline



### PART 1: Initial Setup and Wine Data Preparation

In [2]:
# Load your CSV 
df = pd.read_csv("dataset/wine.csv")
display(df.head())

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735


## Data Preprocessing

Before training the model, the following preprocessing steps are applied:

* The target variable (`Wine`) is separated from the features.
* The dataset is split into **training (80%)** and **testing (20%)** sets.
* **Stratified sampling** is used to preserve class proportions.
* Feature scaling is performed using **StandardScaler**, which is essential for SVMs due to their sensitivity to feature magnitude.

In [3]:
y = df["Wine"]
X = df.drop(columns=["Wine"])

In [ ]:
# Stratified split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Baseline SVM Model

As a baseline, an SVM with an **RBF kernel** is trained using default hyperparameters:

* `kernel = 'rbf'`
* `C = 1.0`
* `gamma = 'scale'`

The model is implemented using a **Pipeline**, ensuring that scaling is applied consistently during both training and inference.

In [5]:
# Define the initial pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])

## Model Evaluation Metrics

The trained model is evaluated using the following metrics:

* **Accuracy**: Overall classification correctness
* **Confusion Matrix**: Class-wise prediction analysis
* **Precision, Recall, F1-Score**: Detailed per-class performance

These metrics provide a comprehensive understanding of the model’s behavior across all wine classes.

In [ ]:
# Fit and evaluate the initial pipeline 
pipeline.fit(X_train, y_train)
y_pred_initial = pipeline.predict(X_test)
acc_initial = accuracy_score(y_test, y_pred_initial)

print(f"Initial SVM Model Accuracy: {acc_initial:.4f}")
print("Initial Classification Report:\n")
print(classification_report(y_test, y_pred_initial))

Initial SVM Model Accuracy: 0.9722
Initial Classification Report:

              precision    recall  f1-score   support

           1       1.00      1.00      1.00        12
           2       0.93      1.00      0.97        14
           3       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



## Hyperparameter Tuning with GridSearchCV


SVM performance is highly sensitive to hyperparameter selection.
To systematically identify the best configuration, **GridSearchCV** is used with **5-fold stratified cross-validation**.

---

### Hyperparameters Explored

#### 1. Regularization Parameter (C)

Controls the trade-off between:

* Maximizing the margin

* Minimizing classification errors

* Small `C` → higher bias, simpler decision boundary

* Large `C` → lower bias, risk of overfitting

#### 2. Kernel Function

* **Linear**: Suitable for linearly separable data
* **RBF**: Captures nonlinear decision boundaries
* **Polynomial**: Models polynomial feature interactions

#### 3. Gamma (for RBF and Polynomial kernels)

Controls how far the influence of a single training point reaches.

* Small `gamma` → smoother decision boundary
* Large `gamma` → complex, highly flexible boundary


### PART 2: Hyperparameter Tuning using GridSearchCV on Wine Data

In [7]:
# Define the parameter grid to search
param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'svm__kernel': ['rbf', 'poly', 'sigmoid']
}

In [8]:
# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,   
    verbose=0
)

In [9]:
# Perform the grid search
grid_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'svm__C': [0.1, 1, ...], 'svm__gamma': ['scale', 'auto', ...], 'svm__kernel': ['rbf', 'poly', ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


## Best Hyperparameters (Wine Dataset)

After performing GridSearchCV, the following configuration achieved the highest cross-validated accuracy:



In [10]:
# Identify the best parameters and score
best_cv_score = grid_search.best_score_
best_params = grid_search.best_params_
best_pipeline = grid_search.best_estimator_

## Final Evaluation on Test Set

Using the best hyperparameters, the optimized SVM model is evaluated on the test set.


In [11]:
# Evaluate the best model on the test set
y_pred_best = best_pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_best)

print("Tuning Results:")
print(f"* Best Cross-Validation Accuracy: {best_cv_score:.4f}")
print(f"* Best Hyperparameters: {best_params}")
print(f"* Test Accuracy with Best Model: {test_accuracy:.4f}\n")

print("Classification Report (Tuned Model):\n")
print(classification_report(y_test, y_pred_best))

Tuning Results:
* Best Cross-Validation Accuracy: 0.9931
* Best Hyperparameters: {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
* Test Accuracy with Best Model: 1.0000

Classification Report (Tuned Model):

              precision    recall  f1-score   support

           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        14
           3       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



#  Breast Cancer Dataset — Model Comparison

## Objective

In this section, we apply:

1. **Support Vector Machine (SVM)**
2. **Random Forest Classifier**

to the **Breast Cancer dataset**, and compare their performance.

### PART 3: Comparison on Breast Cancer Dataset (SVM vs. Random Forest) 

In [12]:
# Data Loading and Verification

path = "dataset/breast-cancer.csv"
if not os.path.exists(path):
    raise FileNotFoundError(f"{path} is not exist")
else:
    print('Data file loaded successfully.')

Data file loaded successfully.


## Dataset Description

The Breast Cancer dataset contains diagnostic features extracted from digitized images of breast tissue.

**Target classes**

* 0 → Malignant
* 1 → Benign

This is a **binary classification problem**, where minimizing false negatives (malignant predicted as benign) is particularly important.

In [ ]:
df = pd.read_csv(path)

print("Dataset Head")
display(df.head())

Dataset Head


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [14]:
# Feature and Target Separation 
y_raw = df["diagnosis"]
X = df.drop(columns=["diagnosis"]) 

In [15]:
# Target Encoding (Malignant/Benign)
le = LabelEncoder()

# M (Malignant) -> 1, B (Benign) -> 0
y = le.fit_transform(y_raw) 
class_names = list(le.classes_)
print(f"Encoded Class Names: {class_names}") 

Encoded Class Names: ['B', 'M']


In [16]:
# Train/Test Split
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )

## Models Used

### 1. Support Vector Machine (SVM)

* Uses feature scaling (StandardScaler)
* Hyperparameters optimized via GridSearchCV
* Probability estimates enabled for ROC-AUC analysis

### 2. Random Forest

* Ensemble of decision trees
* No feature scaling required
* Robust to noise and nonlinearities
* Uses class weighting to handle class imbalance

In [ ]:
# 1. SVM Pipeline  
svm_pipeline_bc = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42))
])

In [18]:
svm_pipeline_bc.fit(X_train_bc, y_train_bc)
y_pred_svm_bc = svm_pipeline_bc.predict(X_test_bc)
svm_accuracy = accuracy_score(y_test_bc, y_pred_svm_bc)

In [ ]:
# 2. Random Forest Classifier 

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_bc, y_train_bc)
y_pred_rf_bc = rf_model.predict(X_test_bc)
rf_accuracy = accuracy_score(y_test_bc, y_pred_rf_bc)

## Evaluation Metrics

Both models are evaluated using:

* Accuracy
* Confusion Matrix
* Precision, Recall, F1-Score
* ROC-AUC score

In [20]:
print("Final Model Comparison:")
print(f" SVM model➡️ Test Accuracy: {svm_accuracy:.4f}")
print(f" Random Forest model➡️ Test Accuracy: {rf_accuracy:.4f}")

print("\n SVM Classification Report (Breast Cancer):")
print(classification_report(y_test_bc, y_pred_svm_bc))

print("\n Random Forest Classification Report (Breast Cancer):")
print(classification_report(y_test_bc, y_pred_rf_bc))

if rf_accuracy > svm_accuracy:
    print("\n➡️ **Conclusion:** Random Forest achieved a slightly higher accuracy on the Breast Cancer test set.")
elif svm_accuracy > rf_accuracy:
    print("\n➡️ **Conclusion:** The SVM model achieved a slightly higher accuracy on the Breast Cancer test set.")
else:
    print("\n➡️ **Conclusion:** Both SVM and Random Forest achieved the same accuracy on the Breast Cancer test set.")

Final Model Comparison:
 SVM model➡️ Test Accuracy: 0.9737
 Random Forest model➡️ Test Accuracy: 0.9561

 SVM Classification Report (Breast Cancer):
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        72
           1       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114


 Random Forest Classification Report (Breast Cancer):
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        72
           1       1.00      0.88      0.94        42

    accuracy                           0.96       114
   macro avg       0.97      0.94      0.95       114
weighted avg       0.96      0.96      0.96       114


➡️ **Conclusion:** The SVM model achieved a slightly higher accuracy on the Breast Cancer test set.


 

## Results Summary

| Model         | Accuracy | ROC-AUC   | Strengths                                         |
| ------------- | -------- | --------- | ------------------------------------------------- |
| SVM           | High     | Very High | Excellent class separation, strong generalization |
| Random Forest | High     | High      | Robust, interpretable feature importance          |
 

## Conclusion

* **SVM** achieves superior performance when properly scaled and tuned.
* **Random Forest** provides a strong and reliable baseline with minimal preprocessing.
* Both models perform well, but SVM often achieves slightly higher ROC-AUC on this dataset.
 
 
